# 4. 使用 Zeo++ 进行孔隙分析（Windows 版）

语言：[English](../en/04_zeopp_pore_analysis.ipynb) | **中文**

本原生 Windows 版 Notebook 会查找 Windows 版 Notebook 2 生成的 TAPB–TFB CIF，通过普通 Python 单元格运行 cofkit 的 Zeo++ 分析封装，并在无需 WSL 的情况下输出便于阅读的孔隙信息。等价的 `analyze_zeopp_pore_properties` Python 工作流以注释代码形式提供。

## 教程系列

1. [环境设置](01_environment_setup.ipynb)
2. [首次构建 COF](02_first_cof_build.ipynb)
3. [拓扑、连接数组合与连接键示例](03_topologies_connectivities_and_linkages.ipynb)
4. **使用 Zeo++ 进行孔隙分析**（本 Notebook）

前置条件：成功运行 Windows 版 [Notebook 2](02_first_cof_build.ipynb)，并准备好可在 Windows 上运行的 Zeo++ 0.3 `network.exe` 或 `network` 可执行文件。`cofkit` Python 软件包本身无需 Zeo++ 即可工作；只有本分析需要这个外部程序。

## 查找 Notebook 2 的 CIF

构建流程会按粗略验证类别对 CIF 分组，因此具体子目录可能不同。下面的搜索会遍历所有类别；如果 Notebook 2 尚未生成 CIF，则会给出明确的错误信息。

In [ ]:
from pathlib import Path

matches = sorted(
    Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif")
)
if not matches:
    raise FileNotFoundError(
        "未找到 TAPB–TFB CIF。请先运行 Windows 版 Notebook 2。"
    )

cif_path = matches[0].resolve()
print(f"使用 CIF： {cif_path}\n")
print("\n".join(cif_path.read_text(encoding="utf-8").splitlines()[:18]))

In [ ]:
# Python 等价实现（已注释）：查找 Notebook 2 的 CIF。
# from pathlib import Path
# matches = sorted(Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# if not matches:
#     raise FileNotFoundError("Run Notebook 2 before continuing.")
# cif_path = matches[0]
# print(cif_path)

## 粘贴 Zeo++ 路径

在下一个代码单元格中，只需把 `请在这里粘贴 network.exe 的完整路径` 替换为教师提供的完整路径。请保留两侧的英文双引号，然后运行该单元格。例如，修改后的代码可以是 `zeopp_path_text = r"C:\course-tools\zeo++-0.3\network.exe"`。开头的 `r` 可以让 Windows 路径中的反斜杠保持原样。

该单元格会检查文件，并把设置保存到仓库本地的 `.env` 文件，因此只需粘贴一次。已有的其他 `.env` 设置会被保留。上游 Zeo++ 0.3 文档说明可通过 Cygwin 和 GNU Make 在 Windows 上编译，但在受限的 Windows 环境中，本 Notebook 有意把安装工作留给教师完成。

In [ ]:
from pathlib import Path
import re

# 学生：请把完整路径粘贴到这个原始字符串中。
zeopp_path_text = r"请在这里粘贴 network.exe 的完整路径"

placeholder = "请在这里粘贴 network.exe 的完整路径"
if zeopp_path_text == placeholder:
    raise ValueError(
        "请把 network.exe 的完整路径粘贴到 zeopp_path_text 中，然后重新运行此单元格。"
    )

zeopp_path = Path(zeopp_path_text).expanduser()
if not zeopp_path.is_file():
    raise FileNotFoundError(f"Zeo++ 可执行文件不存在： {zeopp_path}")
zeopp_path = zeopp_path.resolve()

repo_root = Path.cwd().resolve()
dotenv_path = repo_root / ".env"
existing_lines = (
    dotenv_path.read_text(encoding="utf-8-sig").splitlines()
    if dotenv_path.is_file()
    else []
)
kept_lines = [
    line
    for line in existing_lines
    if not re.match(r"\s*(?:export\s+)?COFKIT_ZEOPP_PATH\s*=", line)
]
kept_lines.append(f'COFKIT_ZEOPP_PATH="{zeopp_path}"')
dotenv_path.write_text("\n".join(kept_lines) + "\n", encoding="utf-8")
print(f"已为本仓库保存 Zeo++ 路径： {zeopp_path}")

In [ ]:
# 在 Windows 上配置或编译 Zeo++ 没有对应的 Python API 实现。
# Zeo++ 是外部 C++ 程序；cofkit 的 Python API 从下方的分析步骤开始。

## 运行点探针和有限尺寸探针分析

基线分析会报告限制孔道的球尺寸，以及点探针可接触表面积和体积。重复使用 `--probe-radius` 会增加考虑可达性的探针扫描；这里示范的半径为 1.20 Å 和 1.86 Å。请根据实际研究所用的吸附质模型调整这些半径。

命令有意省略 `--json`，因此终端会输出简洁易读的报告。同时仍会在磁盘上写入完整的机器可读 `zeopp_report.json`。

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
matches = sorted(
    Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif")
)
if not matches:
    raise FileNotFoundError("请先运行 Windows 版 Notebook 2。")
cif_path = matches[0].resolve()

dotenv_path = repo_root / ".env"
if not dotenv_path.is_file():
    raise FileNotFoundError("请先运行“粘贴 Zeo++ 路径”单元格。")
zeopp_entry = next(
    (
        line
        for line in dotenv_path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip().startswith("COFKIT_ZEOPP_PATH=")
    ),
    None,
)
if zeopp_entry is None:
    raise RuntimeError("请先运行“粘贴 Zeo++ 路径”单元格。")
zeopp_path = Path(
    zeopp_entry.split("=", 1)[1].strip().strip("\"'")
).expanduser()
if not zeopp_path.is_file():
    raise FileNotFoundError(f"Zeo++ 可执行文件不存在： {zeopp_path}")

conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("未找到 Conda。请先运行 Windows 版 Notebook 1。")
command = [
    conda, "run", "-n", "cofkit", "cofkit",
    "analyze", "zeopp", str(cif_path),
    "--zeopp-path", str(zeopp_path.resolve()),
    "--output-dir", "out/tutorial_04_zeopp",
    "--probe-radius", "1.20",
    "--probe-radius", "1.86",
    "--surface-samples-per-atom", "250",
    "--volume-samples-total", "5000",
]
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python API 等价实现（已注释），包括便于阅读的报告。
# from pathlib import Path
# from cofkit import analyze_zeopp_pore_properties
#
# cif_path = next(Path("out/tutorial_02_first_build/cifs").rglob("tapb__tfb__hcb.cif"))
# zeopp_entry = next(
#     line for line in Path(".env").read_text(encoding="utf-8").splitlines()
#     if line.strip().startswith("COFKIT_ZEOPP_PATH=")
# )
# zeopp_binary = zeopp_entry.split("=", 1)[1].strip().strip("\"'")
# result = analyze_zeopp_pore_properties(
#     cif_path,
#     output_dir="out/tutorial_04_zeopp_python",
#     probe_radii=(1.20, 1.86),
#     surface_samples_per_atom=250,
#     volume_samples_total=5000,
#     zeopp_path=zeopp_binary,
# )
# basic = result.baseline.basic_pore_properties
# channels = result.baseline.point_probe_channels
# surface = result.baseline.point_probe_surface_area
# volume = result.baseline.point_probe_volume
# print(f"Input CIF: {result.input_cif}")
# print(f"Largest included sphere: {basic.largest_included_sphere:.3f} Å")
# print(f"Largest free sphere: {basic.largest_free_sphere:.3f} Å")
# print("Largest included sphere along a free path: "
#       f"{basic.largest_included_sphere_along_free_path:.3f} Å")
# print(f"Point-probe channels / pockets: {channels.n_channels} / {channels.n_pockets}")
# print(f"Point-probe accessible surface area: {surface.accessible_surface_area_a2:.3f} Å²")
# print(f"Point-probe accessible volume: {volume.accessible_volume_a3:.3f} Å³")
# for scan in result.probe_scans:
#     print(f"Probe radius {scan.settings.probe_radius:.2f} Å: {scan.status}")
#     if scan.surface_area is not None and scan.volume is not None:
#         print(f"  accessible surface area = {scan.surface_area.accessible_surface_area_a2:.3f} Å²")
#         print(f"  accessible volume fraction = {scan.volume.accessible_volume_fraction:.5f}")
#     if scan.accessibility is not None:
#         print(f"  accessible Voronoi-node fraction = {scan.accessibility.accessible_fraction}")
# print(f"Full report: {result.report_path}")

## 如何解读主要数值

| 报告字段 | 实际含义 |
|---|---|
| 最大内含球 | 能够放入孔隙网络某处的最大球直径 |
| 最大自由球 | 能够穿过周期性孔隙网络的最大球直径 |
| 自由路径上的最大内含球 | 可贯通路径上遇到的最大孔腔尺寸 |
| 可接触表面积 | 在 Zeo++ 几何模型下，所选探针能够到达的表面 |
| 可接触体积/体积分数 | 探针可到达的孔隙体积，以绝对值或相对晶胞比例表示 |
| 通道与口袋 | 连通的传输路径与孤立的可接触区域 |

报告中的所有球尺寸和探针设置均采用以埃为基础的单位。表面积和体积估计使用蒙特卡洛采样，因此在正式的收敛性研究中应提高采样数量。

## 查看保留的报告和原始输出

cofkit 会保留 Zeo++ 输入/输出文件和日志，使解析后的报告仍可追溯和审查。

In [ ]:
import json
from pathlib import Path

output_dir = Path("out/tutorial_04_zeopp")
report_path = output_dir / "zeopp_report.json"
if not report_path.is_file():
    raise FileNotFoundError("请先运行 Zeo++ 分析单元格，再查看输出。")

print("--- 生成的 Zeo++ 文件 ---")
print(*(path.resolve() for path in sorted(output_dir.rglob("*")) if path.is_file()), sep="\n")
print("\n--- zeopp_report.json ---")
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report, indent=2))

In [ ]:
# Python 等价实现（已注释）：读取已保存的报告。
# import json
# from pathlib import Path
# report_path = Path("out/tutorial_04_zeopp/zeopp_report.json")
# report = json.loads(report_path.read_text())
# print(report["baseline"]["basic_pore_properties"])
# for scan in report["probe_scans"]:
#     print(scan["settings"]["probe_radius"], scan["status"])

## 使用这些数值之前

孔隙描述符的意义取决于输入几何结构和原子半径模型。本教程生成的 CIF 仅用于演示；在科研计算中，应先优化并验证框架结构，记录 Zeo++ 版本和半径定义，并检查采样收敛性。有限尺寸探针扫描即使失败，也只会记录在 JSON 报告中，不会丢弃已成功获得的基线结果。